In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import struct
import os

# ==============================================================
# 1) Physik & Gitter (wie bei dir)
# ==============================================================
lam = 800e-9            # Wellenlänge [m]
BEAM_DIAM = 11e-3       # Strahldurchmesser [m]

N = 1024
L = 6 * BEAM_DIAM
dx = L / N
x = (np.arange(N) - N/2) * dx
y = x
X, Y = np.meshgrid(x, y)

# ==============================================================
# 2) WCF-READER (genau wie im anderen Programm)
# ==============================================================
def read_wcf_frames(filepath):
    frames = []
    FILE_HEADER  = 5592
    FRAME_HEADER = 944

    with open(filepath, "rb") as fobj:
        fobj.seek(FILE_HEADER)
        expected_w = None
        expected_h = None

        while True:
            header = fobj.read(FRAME_HEADER)
            if len(header) < FRAME_HEADER:
                break

            w = struct.unpack_from("<I", header, 20)[0]
            h = struct.unpack_from("<I", header, 24)[0]

            if expected_w is None:
                expected_w, expected_h = w, h

            # Frames mit abweichender Größe überspringen
            if w != expected_w or h != expected_h:
                fobj.seek(w*h*2, 1)
                continue

            data = fobj.read(w*h*2)
            if len(data) < w*h*2:
                break

            frames.append(np.frombuffer(data, dtype=np.uint16).reshape((h, w)))

    return np.array(frames)

def extract_2d_profile_from_wcf(filename):
    frames = read_wcf_frames(filename)
    if frames.size == 0:
        raise RuntimeError("Keine gültigen Frames in WCF-Datei!")

    avg = np.mean(frames, axis=0).astype(float)

    # Background abziehen + clip
    bg = np.min(avg)
    avg = avg - bg
    avg[avg < 0] = 0.0

    # Normieren
    max_val = np.max(avg)
    if max_val > 0:
        avg /= max_val

    h, w = avg.shape
    return avg, (h, w)

def make_dataprofile(filename="BeamProfile1.wcf"):
    print(">>> Nutze gemessenes WCF-Profil als 2D-Eingangsfeld")
    if not os.path.exists(filename):
        raise FileNotFoundError(f"{filename} nicht gefunden!")

    profile_wcf, (h_wcf, w_wcf) = extract_2d_profile_from_wcf(filename)
    if (h_wcf, w_wcf) != (N, N):
        raise ValueError(f"WCF-Daten haben Shape {(h_wcf, w_wcf)}, aber Simulation erwartet {(N, N)}.")

    # 1) Zentrieren (damit Maske/Propagation passt)
    I = center_by_centroid(profile_wcf, thresh_rel=0.2)

    # 2) Physikalische Strahlmaske (11 mm Durchmesser)
    R0 = BEAM_DIAM / 2.0
    R = np.sqrt(X**2 + Y**2)
    beam_mask = (R <= R0)

    # 3) Robust Background: Median außerhalb des Strahls (oder Randring)
    outside = ~beam_mask
    bg = np.median(I[outside]) if np.any(outside) else np.percentile(I, 5)

    I = I - bg
    I[I < 0] = 0.0

    # 4) Nur Strahl behalten (Top-Hat soll außerhalb ~0 sein)
    I *= beam_mask.astype(float)

    # 5) Normierung innerhalb der Maske
    peak = I.max()
    if peak > 0:
        I /= peak

    # 6) Intensität -> Feldamplitude
    return np.sqrt(I).astype(complex)

def center_by_centroid(I, thresh_rel=0.2):
    """Zentriert Intensitätsbild per Schwerpunkt oberhalb eines Thresholds (integer shift)."""
    I = np.asarray(I, float)
    t = thresh_rel * I.max() if I.max() > 0 else 0
    m = I > t
    if not np.any(m):
        return I

    yy, xx = np.indices(I.shape)
    cx = (xx[m] * I[m]).sum() / (I[m].sum() + 1e-12)
    cy = (yy[m] * I[m]).sum() / (I[m].sum() + 1e-12)

    # Ziel: Mitte des Arrays
    sx = int(round(I.shape[1] / 2 - cx))
    sy = int(round(I.shape[0] / 2 - cy))
    return np.roll(np.roll(I, sy, axis=0), sx, axis=1)

# ==============================================================
# 3) Angular Spectrum Propagation (wie bei dir)
# ==============================================================
def propagate_asm(U0, z):
    k = 2*np.pi / lam

    fx = np.fft.fftfreq(N, d=dx)
    fy = np.fft.fftfreq(N, d=dx)
    FX, FY = np.meshgrid(fx, fy)

    kx = 2*np.pi * FX
    ky = 2*np.pi * FY
    kz = np.sqrt(k**2 - kx**2 - ky**2 + 0j)

    H = np.exp(1j * kz * z)

    U_k = np.fft.fft2(U0)
    Uz = np.fft.ifft2(U_k * H)
    return Uz

# ==============================================================
# 4) Plot-Funktionen (wie bei dir)
# ==============================================================
def plot_2d(U, title="", zoom_mm=10.0, log=False):
    I = np.abs(U)**2
    I /= I.max() if I.max() > 0 else 1

    x_mm = x * 1e3
    m = np.abs(x_mm) <= zoom_mm
    I_zoom = I[np.ix_(m, m)]

    extent = [-zoom_mm, zoom_mm, -zoom_mm, zoom_mm]

    plt.figure(figsize=(5,4))
    if log:
        plt.imshow(I_zoom, extent=extent, origin="lower",
                   norm=LogNorm(vmin=1e-6, vmax=1), aspect="equal")
    else:
        plt.imshow(I_zoom, extent=extent, origin="lower", aspect="equal")

    plt.xlabel("x [mm]")
    plt.ylabel("y [mm]")
    plt.title(title)
    plt.colorbar(label="norm. intensity")
    plt.tight_layout()
    plt.show()

def plot_1d(U, title="", zoom_mm=10.0, log=False):
    I = np.abs(U)**2
    I /= I.max() if I.max() > 0 else 1

    mid = N//2
    I1d = I[mid, :]
    x_mm = x * 1e3

    m = np.abs(x_mm) <= zoom_mm
    x_plot = x_mm[m]
    I_plot = I1d[m]

    plt.figure(figsize=(6,3))
    plt.plot(x_plot, I_plot)
    if log:
        plt.yscale("log")
        plt.ylim(1e-6, 1)

    plt.xlabel("x [mm]")
    plt.ylabel("norm. intensity")
    plt.title(title)
    plt.grid(True, which="both")
    plt.tight_layout()
    plt.show()

# ==============================================================
# 5) Demo
# ==============================================================
if __name__ == "__main__":

    # statt Supergauss:
    U0 = make_dataprofile("BeamProfile1.wcf")

    z = 3.5
    Uz = propagate_asm(U0, z)

    plot_2d(U0, "U0 (WCF) 2D lin", zoom_mm=10, log=False)
    plot_2d(U0, "U0 (WCF) 2D log", zoom_mm=10, log=True)
    plot_1d(U0, "U0 (WCF) 1D lin", zoom_mm=10, log=False)
    plot_1d(U0, "U0 (WCF) 1D log", zoom_mm=10, log=True)

    plot_2d(Uz, f"U(z={z} m) 2D lin", zoom_mm=10, log=False)
    plot_2d(Uz, f"U(z={z} m) 2D log", zoom_mm=10, log=True)
    plot_1d(Uz, f"U(z={z} m) 1D lin", zoom_mm=10, log=False)
    plot_1d(Uz, f"U(z={z} m) 1D log", zoom_mm=10, log=True)



NameError: name 'tiff' is not defined